# 01 — Data Exploration: Deal Close Propensity

**Sprint 28b** · TDR Deal Inspection App

## Objective
Explore the `Forecast_Page_Opportunities_Magic_SNFv2` dataset to understand:
1. **Label distribution** — How many Won vs Lost vs Open deals? Is there enough labeled data?
2. **Feature landscape** — Which of the 506 fields are useful for ML? What's the null rate?
3. **Candidate features** — Identify the strongest signals for predicting deal close propensity.
4. **Data quality** — Types, cardinality, distributions, outliers.
5. **Go/No-Go** — Is there sufficient labeled data (≥500 closed deals) for `SNOWFLAKE.ML.CLASSIFICATION`?

### Data Sources
| Source | Description |
|--------|-------------|
| `TDR_APP.PUBLIC."Forecast_Page_Opportunities_Magic_SNFv2"` | Full Snowflake table (canonical, v2) |
| `samples/forecast_page_opportunities_cv2.json` | 500-record local sample (v2 schema) |

We start with the local sample for schema inspection, then connect to Snowflake for the full dataset.


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
print('Imports OK')


---
## Part 1: Local Sample — Schema Inspection

Load the 500-record v2 sample to understand field names, types, and null rates.
This sample gives us the full 506-column schema for feature candidate identification.


In [ ]:
# ── Load local sample (v2 dataset) ───────────────────────────────────
with open('../samples/forecast_page_opportunities_cv2.json') as f:
    raw = json.load(f)

df_sample = pd.DataFrame(raw)
print(f'Sample shape: {df_sample.shape[0]:,} rows × {df_sample.shape[1]} columns')
print(f'Memory usage: {df_sample.memory_usage(deep=True).sum() / 1e6:.1f} MB')


In [ ]:
# ── Column-level stats: type, null rate, unique count ────────────────
col_stats = pd.DataFrame({
    'dtype': df_sample.dtypes,
    'non_null': df_sample.notna().sum(),
    'null_pct': (df_sample.isna().sum() / len(df_sample) * 100).round(1),
    'nunique': df_sample.nunique(),
    'sample_value': df_sample.iloc[0].astype(str).str[:50]
})
col_stats = col_stats.sort_values('null_pct')
print(f'\nColumns with 0% nulls: {(col_stats.null_pct == 0).sum()}')
print(f'Columns with >90% nulls: {(col_stats.null_pct > 90).sum()}')
print(f'Columns with >50% nulls: {(col_stats.null_pct > 50).sum()}')
col_stats.head(30)


In [ ]:
# ── Null heatmap: top 60 columns by completeness ────────────────────
fig, ax = plt.subplots(figsize=(16, 8))
top60 = col_stats[col_stats.null_pct == 0].index[:60]
if len(top60) < 30:
    top60 = col_stats.head(60).index
null_matrix = df_sample[top60].isnull().T
sns.heatmap(null_matrix, cbar=False, yticklabels=True, cmap='YlOrRd', ax=ax)
ax.set_title('Null Pattern — Top 60 Most Complete Columns')
ax.set_xlabel('Row Index')
plt.tight_layout()
plt.show()


In [ ]:
# ── Identify the target variable: "Is Won" ──────────────────────────
print('Target variable: Is Won')
print(df_sample['Is Won'].value_counts(dropna=False))
print()
print('Is Closed:')
print(df_sample['Is Closed'].value_counts(dropna=False))
print()
print('NOTE: Check label variation in the local sample.')
print('Full label distribution will be confirmed via Snowflake.')


---
## Part 2: Candidate Feature Identification

From the 506 v2 columns, we identify features likely to predict deal close propensity.
We group them into categories aligned with the Sprint 28 plan.


In [ ]:
# ── Candidate Feature Groups ────────────────────────────────────────

CANDIDATE_FEATURES = {
    # ── Deal Characteristics ──
    'deal_value': [
        'ACV (USD)', 'ACV (USD) Recurring', 'ACV (USD) Non-Recurring',
        'TCV (USD)', 'TCV Recurring', 'TCV Non-Recurring',
        'Amount', 'Amount (USD)',
        'High', 'Likely', 'High Recurring', 'Likely Recurring',
        'Platform Price', 'Professional Services Price',
        'Consulting Hours', 'Education Price', 'Events Price',
    ],
    'deal_type': [
        'Type', 'Record Type', 'Deal Code', 'Standard Deal',
        'Pricing Type', 'Contract Type', 'DownUpsell',
        'Domo Offering/Solution', 'Business Challenge',
        'Starter', 'Free Attribution', 'Had Free', 'Currently Free',
    ],
    'deal_stage': [
        'Stage', 'Stage Age', 'Domo Forecast Category',
        'Is Pipeline', 'Is Closed', 'Is Won',
        'StageDate1 Pre-Pipeline', 'StageDate2 Determine Needs',
        'StageDate3 Demonstrate Value', 'StageDate4 Confirm Solution',
        'StageDate5 Negotiations', 'StageDate Closed Won', 'StageDate Closed Lost',
    ],
    'deal_timing': [
        'Created Date', 'Close Date', 'Close Date FQ', 'Close Date FY',
        'Fiscal Quarter Relative to Current',
        'Discovery Call Completed', 'Discovery Call Scheduled',
        'Firm Date Agreed Date', 'Verbal Agreement Date',
        'Gate Call Completed', 'Demo Completed Date', 'Completed POC Date',
        'Last Activity Date', 'Last Modified Date',
        'Timeline Budget',
    ],

    # ── Account Characteristics ──
    'account_firmographics': [
        'Account Employees', 'Employees',
        'Account Revenue USD', 'Revenue (USD)',
        'Account Revenue Band', 'Revenue Band',
        'Account Type', 'Account Category',
        'Account Status', 'Customer Account Status',
        'Account Vertical', 'Account Sales Vertical',
        'Account Super Region', 'Super Region', 'Region',
        'Industry (Domo)', 'Sales Segment',
    ],
    'account_history': [
        'New Logo Won Count', 'New Logo Lost Count', 'New Logo Pipeline Count',
        'Renewal Won Count', 'Renewal Lost Count', 'Renewal Pipeline Count',
        'Upsell Won Count', 'Upsell Lost Count', 'Upsell Pipeline Count',
        'Total Closed Won Count', 'Total Closed Lost Count',
        'Total Opty Count', 'Total Pipeline Count',
        'Account ACV', 'Total Won ACV', 'Total Opportunity ACV',
        'ARR', 'New ARR', 'Finance ARR',
        'Customer Tenure (Months)', 'Customer Tenure (Years)',
        'Customer Tenure Category', 'Customer Type',
    ],
    'account_engagement': [
        'People AI Engagement Level', 'People AI Engagement Level 2',
        '6sense Account Buying Stage', '6sense Account Intent Score',
        'ABM Status', 'Number of Instances', 'Number of Free Instances',
        'Domains', 'Trial Domains', 'Active Domain Count',
        'Unique Environments',
    ],

    # ── Sales Process ──
    'sales_process': [
        'Key Lead Source', 'Key Lead Source Level 2',
        'Lead Source', 'Lead Type',
        'Opty Creator Role', 'Forecast Owner Tenure',
        'Sales Consultant', 'Lead SC',
        'Has Pre-Call Plan', 'Has ADM/AE Sync Agenda',
        'Number of Competitors', 'Non-Competitive Deal',
        'Number of Deal Desk Reviews',
        'Quotes: Total', 'Consumption Quotes',
        'Line Items', 'CPQ',
        'Partner Influence', 'Partner Role',
        'Targeted Account Reason',
    ],
    'sales_methodology': [
        'Forecast Comments', 'Next Step', 'Business Challenge',
    ],

    # ── Team / Org ──
    'org_structure': [
        'Team', 'Sub-Team', 'Sales Segment',
        'Mgmt Type', 'Front Line Leader',
        'Level 2 Leader', 'Level 3 Leader', 'Level 4 Leader',
    ],
}

print('Feature Groups and Counts:')
total = 0
for group, cols in CANDIDATE_FEATURES.items():
    present = [c for c in cols if c in df_sample.columns]
    missing = [c for c in cols if c not in df_sample.columns]
    total += len(present)
    print(f'  {group:25s}  {len(present):3d} present / {len(cols):3d} listed')
    if missing:
        print(f'    ⚠ Missing: {missing}')
print(f'\n  TOTAL candidate features: {total}')


In [ ]:
# ── Completeness of candidate features ──────────────────────────────
all_candidates = [c for cols in CANDIDATE_FEATURES.values() for c in cols if c in df_sample.columns]
candidate_nulls = (df_sample[all_candidates].isnull().sum() / len(df_sample) * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, max(8, len(all_candidates) * 0.22)))
candidate_nulls.plot.barh(ax=ax, color=[
    '#2ecc71' if v < 10 else '#f39c12' if v < 50 else '#e74c3c' for v in candidate_nulls
])
ax.set_xlabel('Null %')
ax.set_title('Candidate Feature Null Rates (Local Sample)')
ax.axvline(x=50, color='red', linestyle='--', alpha=0.5, label='50% threshold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nFeatures with <10% nulls (GREEN): {(candidate_nulls < 10).sum()}')
print(f'Features with 10-50% nulls (ORANGE): {((candidate_nulls >= 10) & (candidate_nulls < 50)).sum()}')
print(f'Features with >50% nulls (RED): {(candidate_nulls >= 50).sum()}')


In [ ]:
# ── Numeric feature distributions ───────────────────────────────────
numeric_candidates = df_sample[all_candidates].select_dtypes(include=[np.number]).columns.tolist()
print(f'Numeric candidate features: {len(numeric_candidates)}')

df_sample[numeric_candidates].describe().T.sort_values('std', ascending=False).head(20)


In [ ]:
# ── Categorical feature distributions ───────────────────────────────
cat_candidates = df_sample[all_candidates].select_dtypes(include=['object']).columns.tolist()
print(f'Categorical candidate features: {len(cat_candidates)}\n')

for col in cat_candidates:
    vc = df_sample[col].value_counts(dropna=False).head(8)
    null_pct = df_sample[col].isna().mean() * 100
    nunique = df_sample[col].nunique()
    print(f'── {col} (nunique={nunique}, null={null_pct:.0f}%) ──')
    print(vc.to_string())
    print()


In [ ]:
# ── Stage distribution ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df_sample['Stage'].value_counts().sort_index().plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title('Stage Distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

df_sample['Type'].value_counts().plot.bar(ax=axes[1], color='coral')
axes[1].set_title('Opportunity Type Distribution')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


In [ ]:
# ── ACV distribution ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

acv = df_sample['ACV (USD)'].dropna()
acv.plot.hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('ACV (USD) Distribution')
axes[0].set_xlabel('ACV (USD)')
axes[0].axvline(acv.median(), color='red', linestyle='--', label=f'Median: ${acv.median():,.0f}')
axes[0].legend()

acv_log = np.log1p(acv)
acv_log.plot.hist(bins=50, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('log(1 + ACV) Distribution')
axes[1].set_xlabel('log(1 + ACV)')

plt.tight_layout()
plt.show()

print(f'ACV Stats: mean=${acv.mean():,.0f}  median=${acv.median():,.0f}  std=${acv.std():,.0f}  min=${acv.min():,.0f}  max=${acv.max():,.0f}')


In [ ]:
# ── Derived feature preview: Stage velocity ─────────────────────────
# How long each deal spent in each stage (using StageDate columns)
stage_date_cols = [
    'StageDate1 Pre-Pipeline', 'StageDate2 Determine Needs',
    'StageDate3 Demonstrate Value', 'StageDate4 Confirm Solution',
    'StageDate5 Negotiations',
]

for col in stage_date_cols:
    df_sample[f'{col}_dt'] = pd.to_datetime(df_sample[col], unit='ms', errors='coerce')

# Calculate days between stages
stage_transitions = []
for i in range(len(stage_date_cols) - 1):
    from_col = f'{stage_date_cols[i]}_dt'
    to_col = f'{stage_date_cols[i+1]}_dt'
    transition_name = f'Stage {i+1}→{i+2} Days'
    days = (df_sample[to_col] - df_sample[from_col]).dt.days
    stage_transitions.append(days.rename(transition_name))

stage_vel = pd.concat(stage_transitions, axis=1)
print('Stage Velocity (days between transitions):')
stage_vel.describe().T


In [ ]:
# ── Derived feature preview: Account history signals ────────────────
df_sample['_account_win_rate'] = (
    df_sample['Total Closed Won Count'] /
    df_sample['Total Opty Count'].replace(0, np.nan)
).fillna(0)

df_sample['_revenue_per_employee'] = (
    df_sample['Account Revenue USD'] /
    df_sample['Account Employees'].replace(0, np.nan)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df_sample['_account_win_rate'].plot.hist(bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Account Historical Win Rate')
axes[0].set_xlabel('Win Rate')

rpe = df_sample['_revenue_per_employee'].dropna()
rpe[rpe < rpe.quantile(0.95)].plot.hist(bins=40, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Revenue per Employee (< 95th pctile)')
axes[1].set_xlabel('Revenue / Employee')

plt.tight_layout()
plt.show()


In [ ]:
# ── Correlation heatmap: numeric candidate features ─────────────────
numeric_for_corr = [c for c in numeric_candidates if df_sample[c].notna().sum() > len(df_sample) * 0.5]

# Select a reasonable subset for readability
key_numeric = [
    'ACV (USD)', 'ACV (USD) Recurring', 'TCV (USD)',
    'Amount', 'Platform Price', 'Professional Services Price',
    'Stage Age', 'Line Items', 'Consulting Hours',
    'Account Employees', 'Account Revenue USD',
    'People AI Engagement Level',
    'Total Closed Won Count', 'Total Closed Lost Count',
    'Total Opty Count', 'Number of Competitors',
    'Quotes: Total', 'Domains',
    'Forecast Owner Tenure', 'Number of Deal Desk Reviews',
]
key_numeric = [c for c in key_numeric if c in df_sample.columns]

corr = df_sample[key_numeric].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Correlation Matrix — Key Numeric Features')
plt.tight_layout()
plt.show()


In [ ]:
# ── v2 EDA Candidates — Additional columns worth evaluating ──────────
# These unmapped v2 columns may have ML signal (per shaping doc)
V2_EDA_CANDIDATES = [
    'Customer Tenure (Months)', 'Customer Type', 'Revenue (USD)',
    'Revenue Band', 'Account Status', 'Health Grade GPA',
    'Retention Team', 'Forecast Owner Tenure', 'Customer Tenure Category',
    'StageDate1 Pre-Pipeline', 'StageDate2 Determine Needs',
    'StageDate3 Demonstrate Value', 'StageDate4 Confirm Solution',
    'StageDate5 Negotiations',
    'Total Won ACV', 'Cancel Category', 'Cancelled Date',
    '6sense Account Intent Score',
]

present = [c for c in V2_EDA_CANDIDATES if c in df_sample.columns]
missing = [c for c in V2_EDA_CANDIDATES if c not in df_sample.columns]

print(f'v2 EDA Candidates: {len(present)} present / {len(V2_EDA_CANDIDATES)} listed')
if missing:
    print(f'  ⚠ Missing from sample: {missing}')

if present:
    eda_nulls = (df_sample[present].isnull().sum() / len(df_sample) * 100).sort_values()
    print(f'\nNull rates for v2 EDA candidates:')
    for col, pct in eda_nulls.items():
        signal = '🟢' if pct < 10 else '🟡' if pct < 50 else '🔴'
        print(f'  {signal} {col:40s}  {pct:5.1f}% null')

---
## Part 3: Connect to Snowflake — Full Dataset Analysis (v2)

Now we connect to the canonical v2 Snowflake table to:
1. Get the full row count (including historical closed deals)
2. Check `Is Won` / `Is Closed` label distribution
3. Validate we have enough labeled data for `SNOWFLAKE.ML.CLASSIFICATION` training

**Connection:** Uses `~/.snowflake/connections.toml` (SSO / externalbrowser auth).


In [ ]:
# ── Snowflake Connection ────────────────────────────────────────────
import snowflake.connector
from pathlib import Path
import toml

# Load connection config
toml_path = Path.home() / '.snowflake' / 'connections.toml'
config = toml.load(toml_path)

# Use the default connection (or first available)
conn_name = 'default' if 'default' in config else list(config.keys())[0]
conn_cfg = config[conn_name]

print(f'Connection: {conn_name}')
print(f'Account: {conn_cfg.get("account", "?")}')
print(f'User: {conn_cfg.get("user", "?")}')
print(f'Auth: {conn_cfg.get("authenticator", "snowflake")}')

ctx = snowflake.connector.connect(
    account=conn_cfg['account'],
    user=conn_cfg['user'],
    authenticator=conn_cfg.get('authenticator', 'snowflake'),
    database=conn_cfg.get('database', 'TDR_APP'),
    schema=conn_cfg.get('schema', 'PUBLIC'),
    warehouse=conn_cfg.get('warehouse', 'TDR_APP_WH'),
    role=conn_cfg.get('role', None),
)
cur = ctx.cursor()
print('\n✅ Connected to Snowflake')


In [ ]:
# ── Full dataset stats ──────────────────────────────────────────────
cur.execute('SELECT COUNT(*) FROM TDR_APP.PUBLIC."Forecast_Page_Opportunities_Magic_SNFv2"')
total_rows = cur.fetchone()[0]
print(f'Total rows: {total_rows:,}')

# Label distribution
cur.execute('''
    SELECT
        "Is Closed",
        "Is Won",
        COUNT(*) as cnt,
        ROUND(AVG("ACV (USD)"), 2) as avg_acv
    FROM TDR_APP.PUBLIC."Forecast_Page_Opportunities_Magic_SNFv2"
    GROUP BY "Is Closed", "Is Won"
    ORDER BY "Is Closed", "Is Won"
''')
label_dist = cur.fetchall()
print('\nLabel Distribution:')
print(f'{"Is Closed":>12s} {"Is Won":>10s} {"Count":>10s} {"Avg ACV":>12s}')
print('-' * 50)
for row in label_dist:
    print(f'{str(row[0]):>12s} {str(row[1]):>10s} {row[2]:>10,d} ${row[3]:>10,.0f}')


In [ ]:
# ── Stage distribution (full dataset) ───────────────────────────────
cur.execute('''
    SELECT
        "Stage",
        COUNT(*) as cnt,
        SUM(CASE WHEN "Is Won" = 'true' THEN 1 ELSE 0 END) as won,
        SUM(CASE WHEN "Is Closed" = 'true' AND "Is Won" = 'false' THEN 1 ELSE 0 END) as lost,
        ROUND(AVG("ACV (USD)"), 2) as avg_acv
    FROM TDR_APP.PUBLIC."Forecast_Page_Opportunities_Magic_SNFv2"
    GROUP BY "Stage"
    ORDER BY cnt DESC
''')
stage_data = cur.fetchall()
print('Stage Distribution (Full Dataset):')
print(f'{"Stage":>35s} {"Count":>8s} {"Won":>8s} {"Lost":>8s} {"Win%":>8s} {"Avg ACV":>12s}')
print('-' * 85)
for row in stage_data:
    win_pct = (row[2] / (row[2] + row[3]) * 100) if (row[2] + row[3]) > 0 else 0
    print(f'{str(row[0]):>35s} {row[1]:>8,d} {row[2]:>8,d} {row[3]:>8,d} {win_pct:>7.1f}% ${row[4]:>10,.0f}')


In [ ]:
# ── Type distribution (full dataset) ────────────────────────────────
cur.execute('''
    SELECT
        "Type",
        COUNT(*) as cnt,
        SUM(CASE WHEN "Is Won" = 'true' THEN 1 ELSE 0 END) as won,
        SUM(CASE WHEN "Is Closed" = 'true' AND "Is Won" = 'false' THEN 1 ELSE 0 END) as lost,
        ROUND(AVG("ACV (USD)"), 2) as avg_acv
    FROM TDR_APP.PUBLIC."Forecast_Page_Opportunities_Magic_SNFv2"
    GROUP BY "Type"
    ORDER BY cnt DESC
''')
type_data = cur.fetchall()
print('Type Distribution (Full Dataset):')
print(f'{"Type":>25s} {"Count":>8s} {"Won":>8s} {"Lost":>8s} {"Win%":>8s} {"Avg ACV":>12s}')
print('-' * 75)
for row in type_data:
    win_pct = (row[2] / (row[2] + row[3]) * 100) if (row[2] + row[3]) > 0 else 0
    print(f'{str(row[0]):>25s} {row[1]:>8,d} {row[2]:>8,d} {row[3]:>8,d} {win_pct:>7.1f}% ${row[4]:>10,.0f}')


In [ ]:
# ── Pull labeled data (closed deals) for feature analysis ───────────
cur.execute('''
    SELECT *
    FROM TDR_APP.PUBLIC."Forecast_Page_Opportunities_Magic_SNFv2"
    WHERE "Is Closed" = 'true'
''')
cols = [desc[0] for desc in cur.description]
closed_data = cur.fetchall()

df_closed = pd.DataFrame(closed_data, columns=cols)
df_closed['label'] = (df_closed['Is Won'] == 'true').astype(int)

print(f'Closed deals loaded: {len(df_closed):,}')
print(f'Won: {df_closed.label.sum():,}  Lost: {(1 - df_closed.label).sum():,}')
print(f'Win rate: {df_closed.label.mean():.1%}')
print(f'\nClass balance (Won:Lost ratio): 1:{(1 - df_closed.label.mean()) / max(df_closed.label.mean(), 0.001):.1f}')


In [ ]:
# ── Feature importance preview: univariate relationship to label ────
# Quick test: do any numeric features significantly differ between Won and Lost?

key_cols = [
    'ACV (USD)', 'ACV (USD) Recurring', 'TCV (USD)',
    'Stage Age', 'Line Items', 'Consulting Hours',
    'Account Employees', 'Account Revenue USD',
    'People AI Engagement Level',
    'Total Closed Won Count', 'Total Closed Lost Count',
    'Total Opty Count', 'Number of Competitors',
    'Forecast Owner Tenure', 'Number of Deal Desk Reviews',
    'Domains',
]

# Filter to columns that exist in the Snowflake result set
avail_cols = [c for c in key_cols if c in df_closed.columns]

if len(df_closed) > 0:
    won = df_closed[df_closed.label == 1]
    lost = df_closed[df_closed.label == 0]

    print(f'{"Feature":>35s} {"Won Mean":>12s} {"Lost Mean":>12s} {"Diff%":>10s} {"Direction":>12s}')
    print('-' * 85)
    for col in avail_cols:
        try:
            w_mean = pd.to_numeric(won[col], errors='coerce').mean()
            l_mean = pd.to_numeric(lost[col], errors='coerce').mean()
            if l_mean != 0:
                diff_pct = ((w_mean - l_mean) / abs(l_mean)) * 100
            else:
                diff_pct = 0
            direction = '⬆ Won higher' if w_mean > l_mean else '⬇ Lost higher'
            print(f'{col:>35s} {w_mean:>12,.1f} {l_mean:>12,.1f} {diff_pct:>+9.1f}% {direction:>12s}')
        except Exception:
            pass


In [ ]:
# ── Visualize Won vs Lost distributions for top features ────────────
if len(df_closed) == 0:
    print('⚠️  SKIPPED: No closed deals in dataset — cannot compare Won vs Lost distributions.')
    print('   The table contains only open pipeline. We need historical closed deals for training.')
    print('   → ACTION: Identify a Snowflake table or Domo dataset that includes Is Closed = true.')
else:
    plot_features = [c for c in ['ACV (USD)', 'Stage Age', 'Account Employees',
                                  'People AI Engagement Level', 'Total Opty Count',
                                  'Forecast Owner Tenure'] if c in df_closed.columns]

    n_plots = len(plot_features)
    fig, axes = plt.subplots(2, min(3, (n_plots + 1) // 2), figsize=(18, 10))
    axes = axes.flatten()

    for i, col in enumerate(plot_features):
        if i >= len(axes):
            break
        data_won = pd.to_numeric(won[col], errors='coerce').dropna()
        data_lost = pd.to_numeric(lost[col], errors='coerce').dropna()

        # Clip at 95th percentile for readability
        all_vals = pd.concat([data_won, data_lost])
        clip_val = all_vals.quantile(0.95)
        if clip_val > 0:
            data_won = data_won.clip(upper=clip_val)
            data_lost = data_lost.clip(upper=clip_val)

        axes[i].hist(data_won, bins=30, alpha=0.6, label='Won', color='#2ecc71', density=True)
        axes[i].hist(data_lost, bins=30, alpha=0.6, label='Lost', color='#e74c3c', density=True)
        axes[i].set_title(col)
        axes[i].legend()

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle('Feature Distributions: Won vs Lost', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Categorical feature win rates ───────────────────────────────────
if len(df_closed) == 0:
    print('⚠️  SKIPPED: No closed deals — cannot compute categorical win rates.')
else:
    cat_features = ['Type', 'Stage', 'Sales Segment', 'Record Type',
                    'Key Lead Source', 'Pricing Type', 'Deal Code',
                    'Account Super Region', 'Domo Offering/Solution']
    cat_features = [c for c in cat_features if c in df_closed.columns]

    for col in cat_features:
        print(f'\n── {col} ──')
        ct = df_closed.groupby(col)['label'].agg(['count', 'sum', 'mean']).rename(
            columns={'count': 'Total', 'sum': 'Won', 'mean': 'Win Rate'}
        ).sort_values('Total', ascending=False).head(10)
        ct['Win Rate'] = ct['Win Rate'].map('{:.1%}'.format)
        print(ct.to_string())


---
## Part 4: Data Quality Summary & Recommendations

Run this cell after executing the full notebook to produce a summary.


In [ ]:
# ── Summary Report ──────────────────────────────────────────────────
n_closed = len(df_closed)
n_won = int(df_closed.label.sum()) if n_closed > 0 else 0
n_lost = n_closed - n_won
win_rate = df_closed.label.mean() if n_closed > 0 else 0

print('='*70)
print('  DATA EXPLORATION SUMMARY — Deal Close Propensity')
print('='*70)
print(f'''
LOCAL SAMPLE
  Records:      {len(df_sample):,}
  Columns:      {df_sample.shape[1]}
  Status:       Open pipeline only (no closed deals)

SNOWFLAKE (Full Dataset)
  Table:        TDR_APP.PUBLIC."Forecast_Page_Opportunities_Magic_SNFv2"
  Total Rows:   {total_rows:,}
  Closed Deals: {n_closed:,}
  Won:          {n_won:,}
  Lost:         {n_lost:,}
  Win Rate:     {win_rate:.1%}

LABEL VIABILITY
  Minimum required:     500 labeled (Won+Lost)
  Available:            {n_closed:,}
  Status:               {"✅ SUFFICIENT for SNOWFLAKE.ML.CLASSIFICATION" if n_closed >= 500 else "⚠️ MARGINAL" if n_closed >= 200 else "❌ INSUFFICIENT — need ≥100 closed deals minimum"}

CANDIDATE FEATURES (from local sample schema)
  Total identified:     {len(all_candidates)}
  Numeric:              {len(numeric_candidates)}
  Categorical:          {len(cat_candidates)}
  <10% null:            {(candidate_nulls < 10).sum()}
  >50% null (drop):     {(candidate_nulls >= 50).sum()}
''')

if n_closed == 0:
    print('🚨 CRITICAL FINDING:')
    print('   This table contains ONLY open pipeline deals (Is Closed = false for all rows).')
    print('   There are NO historical won or lost deals to use as training labels.')
    print()
    print('   ACTION REQUIRED:')
    print('   1. Find a Snowflake table or Domo dataset that includes closed deals')
    print('      (Is Won = true/false where Is Closed = true)')
    print('   2. This may be a different view/table in the same database, or')
    print('      a separate historical SFDC export that includes closed-won + closed-lost opps')
    print('   3. Once identified, re-run this notebook with the correct data source')
else:
    print('NEXT STEPS')
    print('  → Notebook 02: Feature Engineering (derive 19 features per Sprint 28 plan)')
    print('  → Notebook 03: Model Prototyping (train ensemble, evaluate, SHAP explanations)')
print('='*70)


In [ ]:
# ── Save DataFrames for downstream notebooks ────────────────────────
if len(df_closed) > 0:
    df_closed.to_parquet('../notebooks/closed_deals.parquet', index=False)
    print(f'Saved {len(df_closed):,} closed deals to notebooks/closed_deals.parquet')
else:
    print('⚠️  Skipping closed_deals.parquet — no closed deals to save.')

# Save all deals (open pipeline) for feature engineering / schema reference
try:
    cur.execute('SELECT * FROM TDR_APP.PUBLIC."Forecast_Page_Opportunities_Magic_SNFv2"')
    all_cols = [desc[0] for desc in cur.description]
    all_data = cur.fetchall()
    df_all = pd.DataFrame(all_data, columns=all_cols)
    df_all.to_parquet('../notebooks/all_deals.parquet', index=False)
    print(f'Saved {len(df_all):,} total deals to notebooks/all_deals.parquet')
except Exception as e:
    print(f'⚠️  Could not save all_deals.parquet: {e}')

# Clean up
try:
    cur.close()
    ctx.close()
    print('\nSnowflake connection closed.')
except Exception:
    print('\nSnowflake connection already closed.')
